In [2]:
import os
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

DISGENET_FILE = "curated_gene_disease_associations.txt"
OUTPUT_DIR = "Bootcamp/project"
K_NEIGHBORS = 10

os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(
    DISGENET_FILE,
    sep="\t",
    dtype=str,
    low_memory=False
)

score_col = None

for c in ["score", "Score", "score "]:
    if c in df.columns:
        score_col = c
        break

if score_col is None:
    for c in df.columns:
        try:
            df[c].astype(float)
            score_col = c
            break
        except:
            pass

if score_col is None:
    raise ValueError("Score column not found.")

df[score_col] = df[score_col].astype(float)

df = df.dropna(
    subset=["diseaseId", "geneSymbol"]
)

diseases = df["diseaseId"].unique().tolist()
genes = df["geneSymbol"].unique().tolist()

disease2idx = {
    disease: i
    for i, disease in enumerate(diseases)
}

gene2idx = {
    gene: i
    for i, gene in enumerate(genes)
}

idx2disease = {
    i: disease
    for disease, i in disease2idx.items()
}

idx2gene = {
    i: gene
    for gene, i in gene2idx.items()
}

N_d = len(diseases)
N_g = len(genes)

features = np.zeros(
    (N_d, N_g),
    dtype=np.float32
)

for _, row in df.iterrows():

    disease_index = disease2idx[row["diseaseId"]]
    gene_index = gene2idx[row["geneSymbol"]]
    score = row[score_col]

    if score > features[disease_index, gene_index]:
        features[disease_index, gene_index] = score

row_max = features.max(
    axis=1,
    keepdims=True
)

row_max[row_max == 0] = 1

features = features / row_max

np.save(
    os.path.join(
        OUTPUT_DIR,
        "disease_gene_features.npy"
    ),
    features
)

with open(
    os.path.join(
        OUTPUT_DIR,
        "disease2idx.json"
    ),
    "w"
) as f:
    json.dump(disease2idx, f)

with open(
    os.path.join(
        OUTPUT_DIR,
        "gene2idx.json"
    ),
    "w"
) as f:
    json.dump(gene2idx, f)

knn_model = NearestNeighbors(
    n_neighbors=K_NEIGHBORS + 1,
    metric="cosine",
    algorithm="brute"
)

knn_model.fit(features)

model_path = os.path.join(
    OUTPUT_DIR,
    "disease_knn_model.joblib"
)

joblib.dump(
    knn_model,
    model_path
)

disease_names_map = (
    df[["diseaseId", "diseaseName"]]
    .drop_duplicates("diseaseId")
    .set_index("diseaseId")["diseaseName"]
    .to_dict()
)

def find_similar_diseases(
    disease_id,
    top_k=10,
    top_genes=5
):

    if disease_id not in disease2idx:
        raise ValueError(
            f"Disease {disease_id} not found."
        )

    disease_index = disease2idx[disease_id]

    disease_vector = features[
        disease_index
    ].reshape(1, -1)

    distances, indices = knn_model.kneighbors(
        disease_vector,
        n_neighbors=top_k + 1
    )

    target_genes = set(
        np.where(
            features[disease_index] > 0
        )[0]
    )

    results = []

    for distance, index in zip(
        distances[0],
        indices[0]
    ):

        if index == disease_index:
            continue

        similarity = 1 - distance
        similar_disease_id = idx2disease[index]

        gene_scores = features[index]

        top_gene_indices = (
            gene_scores
            .argsort()[-top_genes:][::-1]
        )

        genes_for_disease = [
            (
                idx2gene[i],
                float(gene_scores[i])
            )
            for i in top_gene_indices
            if gene_scores[i] > 0
        ]

        similar_genes = set(
            np.where(
                features[index] > 0
            )[0]
        )

        shared_genes = [
            idx2gene[i]
            for i in target_genes & similar_genes
        ]

        results.append({
            "disease_id": similar_disease_id,
            "disease_name": disease_names_map.get(
                similar_disease_id,
                "Unknown"
            ),
            "similarity": float(similarity),
            "top_genes": genes_for_disease,
            "shared_genes": shared_genes
        })

        if len(results) == top_k:
            break

    return results


target_disease = "umls:C0011860"

results = find_similar_diseases(
    target_disease,
    top_k=10,
    top_genes=5
)

print("=" * 60)
print("EXAMPLE INPUT")
print("=" * 60)

print("Disease ID:", target_disease)
print(
    "Disease Name:",
    disease_names_map.get(
        target_disease,
        "Unknown"
    )
)

print("\n" + "=" * 60)
print("EXAMPLE OUTPUT")
print("=" * 60)

for i, result in enumerate(results, 1):

    print(f"\n{i}. {result['disease_name']}")
    print("   Disease ID:", result["disease_id"])
    print(
        "   Similarity:",
        f"{result['similarity']:.4f}"
    )

    print("   Top genes:")

    for gene, score in result["top_genes"]:
        print(
            f"      {gene}: {score:.4f}"
        )

    print("   Shared genes:")

    if result["shared_genes"]:
        print(
            "      ",
            ", ".join(
                result["shared_genes"][:20]
            )
        )
    else:
        print("      None")

with open(
    os.path.join(
        OUTPUT_DIR,
        "example_results.json"
    ),
    "w"
) as f:
    json.dump(
        {
            "input_disease": target_disease,
            "input_disease_name": disease_names_map.get(
                target_disease,
                "Unknown"
            ),
            "results": results
        },
        f,
        indent=4
    )

print("\nModel saved to:", model_path)
print(
    "Results saved to:",
    os.path.join(
        OUTPUT_DIR,
        "example_results.json"
    )
)

EXAMPLE INPUT
Disease ID: umls:C0011860
Disease Name: Diabetes Mellitus, Type 2

EXAMPLE OUTPUT

1. Diabetes Mellitus, Experimental
   Disease ID: umls:C0011853
   Similarity: 0.4067
   Top genes:
      SLC9A3: 1.0000
      IL1B: 1.0000
      PTGS2: 1.0000
      VEGFA: 1.0000
      UCP2: 1.0000
   Shared genes:
       SOD2, IRS1, TGFB1, KCNJ11, HK1, EDN1, CPT1A, UCP2, GPX1, VEGFA, S100A6, PPARGC1A, PCSK2, SOD1, NKX6-1, ATF3, TNF, ATP2A3, CYBA, PAX6

2. Insulin Resistance
   Disease ID: umls:C0021655
   Similarity: 0.3050
   Top genes:
      INSR: 1.0000
      ADIPOQ: 0.6699
      TNF: 0.6521
      ADRB2: 0.6166
      SIRT1: 0.6032
   Shared genes:
       IRS1, PPARG, EGFR, HMGA1, LEP, NOS3, PPARA, INS, KCNJ11, HMOX1, TNF, C3, SLC2A4, LEPR, INSR, RETN, ADIPOQ, LIPC

3. Hyperinsulinism
   Disease ID: umls:C0020459
   Similarity: 0.2725
   Top genes:
      INSR: 1.0000
      INS: 0.8765
      LEP: 0.7035
      GLUD1: 0.6799
      MC4R: 0.6711
   Shared genes:
       IRS1, LEP, NOS3, INS, 